In [35]:
import os
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
from copy import deepcopy


# Helper constants and file paths (adjust as needed for exam setup)
IMAGE_SIZE = (128, 128)         # width, height for resizing
HOG_ORIENT = 9                 # number of orientation bins for HOG
HOG_PIXELS_PER_CELL = (8, 8)   # size of a cell
HOG_CELLS_PER_BLOCK = (3, 3)   # number of cells per block

# Paths to training/test image directories (to be filled with actual paths)
TRAIN_FACE_DIR = "Subiect3/data/ex1"
ADNOTATION = "Subiect3/data/ex1/adnotari.txt"
# TRAIN_NONFACE_DIR = "path/to/test/nonface_images"
# TEST_FACE_DIR = "path/to/test/face_images"
# TEST_NONFACE_DIR = "path/to/test/nonface_images"


def mirror_image(image):
    new_image = cv2.flip(image, 1)
    
    return new_image


In [36]:
def load_images_and_labels_from_folder(folder, adnot_file, image_size, augment=False):
	images, labels = [], []
	# Read annotation file for labels
	with open(adnot_file, 'r') as f:
		lines = f.readlines()
	# Each line: filename label
	label_dict = {}
	for line in lines:
		parts = line.strip().split()
		if len(parts) >= 2:
			label_dict[parts[0]] = parts[1]
	for filename in os.listdir(folder):
		if not filename.endswith('.png'):
			continue
		img_path = os.path.join(folder, filename)
		img = cv2.imread(img_path)
		if img is None:
			continue
		img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
		img = cv2.resize(img, image_size)
		images.append(img)
		labels.append(label_dict.get(filename, "Unknown"))
		if augment:
			img_mirror = cv2.flip(img, 1)
			images.append(img_mirror)
			labels.append(label_dict.get(filename, "Unknown"))
	return images, labels

# Load testing images and labels
faces_train, labels_faces = load_images_and_labels_from_folder(TRAIN_FACE_DIR, ADNOTATION, IMAGE_SIZE, augment=True)

tommy_images = []
ora_images = []
louie_images = []
andy_images = []

tommy_images_train = []
tommy_images_test = []

ora_images_train = []
ora_images_test = []

louie_images_train = []
louie_images_test = []

andy_images_train = []
andy_images_test = []



for img, label in zip(faces_train, labels_faces):
	if label in ["Tommy"]:
		tommy_images.append(img)
		tommy_images_train.append(img)
		tommy_images_train.append(mirror_image(img))
		tommy_images_test.append(img)
	if label in ["Louie"]:
		louie_images.append(img)
		louie_images_train.append(img)
		louie_images_train.append(mirror_image(img))
		louie_images_test.append(img)
	if label in ["Ora"]:
		ora_images.append(img)
		ora_images_train.append(img)
		ora_images_train.append(mirror_image(img))
		ora_images_test.append(img)
	if label in ["Andy"]:
		andy_images.append(img)
		andy_images_train.append(img)
		andy_images_train.append(mirror_image(img))
		andy_images_test.append(img)



In [37]:
def extract_hog_features(images, orientations, pixels_per_cell, cells_per_block):
    """
    Extract HOG descriptors for a list of grayscale images.
    Returns an array of shape (n_samples, n_features).
    """
    features = []
    for img in images:
        fd = hog(img, orientations=orientations, 
                 pixels_per_cell=pixels_per_cell, cells_per_block=cells_per_block,
                 block_norm='L2-Hys', transform_sqrt=True, feature_vector=True)
        features.append(fd)
    return np.array(features, dtype=np.float32)

# Compute HOG features for train and test sets
# X_train = extract_hog_features(faces_train, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)

#acording to the adnotation

features_tommy_train = extract_hog_features(tommy_images_train, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)
features_ora_train = extract_hog_features(ora_images_train, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)
features_andy_train = extract_hog_features(andy_images_train, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)
features_louie_train = extract_hog_features(louie_images_train, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)

features_tommy_test = extract_hog_features(tommy_images_test, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)
features_ora_test = extract_hog_features(ora_images_test, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)
features_andy_test = extract_hog_features(andy_images_test, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)
features_louie_test = extract_hog_features(louie_images_test, HOG_ORIENT, HOG_PIXELS_PER_CELL, HOG_CELLS_PER_BLOCK)



In [ ]:
def train_classifier(training_examples, training_labels) -> LinearSVC:
        best_accuracy = 0
        best_model = None

        Cs = [10 ** -1,10 ** 0]

        for C in tqdm(Cs):
            model = LinearSVC(C=C)
            model.fit(training_examples, training_labels)
            current_accuracy = model.score(training_examples, training_labels)

            if current_accuracy > best_accuracy:
                best_accuracy = current_accuracy
                best_model = deepcopy(model)


        scores = best_model.decision_function(training_examples)
        positive_scores = scores[training_labels > 0]
        negative_scores = scores[training_labels <= 0]


        plt.plot(np.sort(positive_scores))
        plt.plot(np.zeros(len(positive_scores)))
        plt.plot(np.sort(negative_scores))
        plt.xlabel('Nr example antrenare')
        plt.ylabel('Scor clasificator patrat')
        plt.title('Distributia scorurilor clasificatorului pe exemplele de antrenare')
        plt.legend(['Scoruri exemple pozitive', '0', 'Scoruri exemple negative'])
        plt.show()

        return best_model

positives = np.concatenate((features_tommy_train, features_ora_train))
negatives = np.concatenate((features_louie_train, features_andy_train))

training_examples = np.concatenate((positives, negatives))
training_labels = np.array([1] * len(positives) + [0] * len(negatives))



model = train_classifier(training_examples, training_labels)
#stiu ca da aici eroare
#nu am mai stiut ce sa fac


  0%|          | 0/2 [00:00<?, ?it/s]


ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [40]:
test_descriptors = np.concatenate((features_andy_test, features_ora_test,features_louie_test, features_tommy_test))
expected_positives = np.array([1] * (len(test_descriptors) // 2))
expected_negatives = np.array([0] * (len(test_descriptors) // 2))
expected_labels = np.concatenate((expected_positives, expected_negatives))

num_expected = len(expected_labels)

predictions = model.predict(test_descriptors)
#Se calculeaza cate predictii din cele 60 sunt corecte
num_positive_predictions = np.sum(expected_labels == predictions)

accuracy = num_positive_predictions / num_expected

print(accuracy)

AttributeError: 'NoneType' object has no attribute 'predict'